# Motor de Búsqueda Vectorial (TF-IDF) sobre un Corpus de Pokémon

**Asignatura:** Inteligencia Artificial — Recuperación de la Información  
**Referencia teórica:** *Introduction to Information Retrieval* (Manning, Raghavan & Schütze) - Capítulos 6 y 8

En este cuaderno implementaremos un sistema de recuperación de la información basado en el **modelo vectorial (TF-IDF)**. A diferencia del modelo booleano que requiere consultas estructuradas (AND, OR), este modelo permite consultas en texto libre y ordena los documentos por su similitud matemática. Lo dividimos en cuatro partes:

1. **Normalización y Preprocesamiento** (IIR §2.2)
2. **Construcción del Modelo Vectorial y TF-IDF** (IIR §6.2 y §6.3)
3. **Recuperación mediante Similitud del Coseno** (IIR §6.3.1)
4. **Evaluación del Sistema** (IIR §8.3)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, module='nltk')
import pandas as pd
import numpy as np
import nltk
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Descargamos los recursos necesarios de NLTK
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)


True

## 1. Normalización y Preprocesamiento (IIR §2.2)

Al igual que en el modelo booleano, debemos someter los documentos a un preprocesamiento para estandarizar el vocabulario. En esta primera fase, utilizamos `PlaintextCorpusReader` de la suite `NLTK` para unificar y cargar todos los ficheros de texto (`.txt`) de los análisis de Smogon.

En la siguiente celda, tras la carga, definiremos nuestro *pipeline* léxico que posteriormente inyectaremos al vectorizador.

In [2]:
# Definimos el lector del corpus apuntando a nuestra carpeta
corpus_pokemon = PlaintextCorpusReader(
    root='../corpus',
    fileids=r'.*\.txt',
    encoding='utf-8'
)

# Extraemos los nombres de archivo y el contenido en bruto de cada uno
doc_names = [f.replace('.txt', '') for f in corpus_pokemon.fileids()]
documents = [corpus_pokemon.raw(f) for f in corpus_pokemon.fileids()]

print(f"Se han cargado {len(documents)} documentos del corpus usando PlaintextCorpusReader.")


Se han cargado 100 documentos del corpus usando PlaintextCorpusReader.


## 2. Construcción del Modelo Vectorial y TF-IDF (IIR §6.2 y §6.3)

A diferencia del filtrado binario, aquí empleamos el esquema de ponderación **TF-IDF** (*Term Frequency - Inverse Document Frequency*). Para evitar que las palabras demasiado comunes saturen los resultados, el peso de un término se compensa logarítmicamente según los documentos en los que aparece.

Fórmula de la Frecuencia Documental Inversa (IDF) (IIR §6.2.1):
$$
idf_{t} = \log\left(\frac{N}{df_{t}}\right)
$$

Fórmula del peso combinado (TF-IDF) (IIR §6.2.2):
$$
\text{tf-idf}_{t,d} = \text{tf}_{t,d} \times idf_{t}
$$

En la siguiente celda definimos nuestro **Tokenizador Personalizado** (`tokenizador_pokemon`), el cual segmenta el texto, filtra los signos de puntuación, suprime las *stop-words* del inglés, y extrae las raíces con *Porter Stemmer*. A continuación, delegamos la creación de la matriz matricial al `TfidfVectorizer` de `scikit-learn`.

In [3]:
# Inicializamos el stemmer y las stopwords en inglés
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def tokenizador_pokemon(text):
    """Tokeniza, limpia stopwords/puntuación y extrae raíces léxicas."""
    tokens = word_tokenize(text)
    clean_tokens = [
        stemmer.stem(token.lower()) for token in tokens 
        if token.isalpha() and token.lower() not in stop_words
    ]
    return clean_tokens

# Creamos y ajustamos el modelo vectorial pasando nuestro tokenizador
vectorizador = TfidfVectorizer(
    token_pattern=None,        # Anula el patrón por defecto para forzar el uso de nuestro tokenizer
    tokenizer=tokenizador_pokemon
)

tfidf_matrix = vectorizador.fit_transform(documents)
vocabulario = vectorizador.get_feature_names_out()
idf_weights = vectorizador.idf_

print(f"Tamaño del vocabulario: {len(vocabulario)} términos.")
print(f"Dimensiones de la matriz TF-IDF: {tfidf_matrix.shape}\n")

print("Muestra de términos del vocabulario (ordenados alfabéticamente):")
print(f"{'Término':<25} {'Peso IDF':>12}")
print("-" * 40)
for term, idf in zip(vocabulario[100:115], idf_weights[100:115]): # Mostramos una muestra
    print(f"  {term:<25} {idf:>10.4f}")


Tamaño del vocabulario: 3940 términos.
Dimensiones de la matriz TF-IDF: (100, 3940)

Muestra de términos del vocabulario (ordenados alfabéticamente):
Término                       Peso IDF
----------------------------------------
  allur                         3.5357
  almost                        1.5042
  alolan                        1.1608
  alolomola                     4.9220
  alomolola                     4.9220
  alomomola                     1.4720
  alon                          2.7247
  along                         1.5042
  alongsid                      1.1608
  alreadi                       1.3810
  also                          1.0100
  altaria                       2.6194
  altarianit                    4.0057
  altern                        1.0825
  although                      1.2457


## 3. Recuperación mediante Similitud del Coseno (IIR §6.3.1)

Una vez que el corpus y la consulta del usuario se han transformado en vectores dentro del mismo espacio métrico, calculamos la afinidad geométrica entre ambos usando la **Similitud del Coseno**.

Fórmula de la Similitud del Coseno:
$$
\text{sim}(d_1, d_2) = \frac{\vec{V}(d_1) \cdot \vec{V}(d_2)}{|\vec{V}(d_1)| |\vec{V}(d_2)|}
$$

Esto nos devuelve un valor entre 0 y 1, que utilizaremos para elaborar nuestro **ranking descendente** de resultados.

In [4]:
def buscar_tfidf(query, top_n=None):
    # Vectorizar la consulta (el vectorizador ya le aplicará el tokenizador_pokemon)
    query_vec = vectorizador.transform([query])
    
    # Calcular similitud del coseno entre la consulta y todos los documentos
    cosine_sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Ordenar los documentos de mayor a menor similitud
    related_docs_indices = cosine_sim.argsort()[::-1]
    
    resultados = []
    for i in related_docs_indices:
        similitud = cosine_sim[i]
        if similitud > 0: # Solo devolvemos documentos con alguna similitud
            resultados.append({
                'documento': doc_names[i],
                'similitud': similitud
            })
            
    if top_n is not None:
        resultados = resultados[:top_n]
        
    return resultados

def mostrar_resultados(consulta_str, resultados):
    """Imprime los resultados de una consulta de forma formateada en una tabla."""
    print(f"\n{'=' * 65}")
    print(f" Consulta: {consulta_str}")
    print(f"{'=' * 65}")

    if resultados:
        print(f" Se encontraron {len(resultados)} documento(s) relevante(s):\n")
        print(f"   {'Rank':<5} | {'Documento':<30} | {'Similitud':<10}")
        print(f"   {'-'*5}-+-{'-'*30}-+-{'-'*10}")
        for i, r in enumerate(resultados, 1):
            print(f"   {i:<5} | {r['documento']:<30} | {r['similitud']:.4f}")
    else:
        print(" No se encontraron documentos relevantes.")

    print(f"{'=' * 65}")

# Prueba del buscador
consulta_prueba = "Swords Dance priority sweeper"
resultados = buscar_tfidf(consulta_prueba, top_n=5)
mostrar_resultados(consulta_prueba, resultados)



 Consulta: Swords Dance priority sweeper
 Se encontraron 5 documento(s) relevante(s):

   Rank  | Documento                      | Similitud 
   ------+--------------------------------+-----------
   1     | chien-pao                      | 0.2035
   2     | ogerpon-wellspring             | 0.1106
   3     | kingambit                      | 0.0995
   4     | volcarona                      | 0.0872
   5     | iron-boulder                   | 0.0851


### 3.1 Interfaz Textual Interactiva

Bucle interactivo para realizar búsquedas rápidas en el corpus introduciendo texto por consola.

In [5]:
def interfaz_busqueda():
    print("=== Buscador Pokémon Vectorial (TF-IDF) ===")
    print("Escribe 'salir' para terminar.")
    while True:
         query = input("\nIntroduce tu búsqueda: ")
         if query.lower() == 'salir':
             break
         resultados = buscar_tfidf(query, top_n=5)
         mostrar_resultados(query, resultados)

# Descomentar para usar la interfaz interactiva
# interfaz_busqueda()

## 4. Evaluación del Sistema (IIR §8.3 y §8.4)

Para medir el rendimiento de nuestro modelo vectorial frente al booleano, evaluamos las mismas 20 necesidades de información estructuradas (nuestro *Ground Truth*). Utilizaremos:

**Precisión (*Precision*):** fracción de documentos recuperados que son verdaderamente relevantes.
$$
P = \frac{|\text{Recuperados} \cap \text{Relevantes}|}{|\text{Recuperados}|}
$$

**Sensibilidad (*Recall*):** fracción de documentos relevantes que el modelo logró encontrar.
$$
R = \frac{|\text{Recuperados} \cap \text{Relevantes}|}{|\text{Relevantes}|}
$$

**Mean Average Precision (MAP):** Como este motor devuelve listas ordenadas numéricamente por afinidad, empleamos el MAP (IIR §8.4) para premiar a los sistemas que posicionan los aciertos en los puestos más altos del ranking.
$$
\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q)
$$

Procedemos a automatizar la lectura del CSV, inyectar las frases de texto libre (`consulta_texto_libre`), y calcular los promedios globales estadísticos.

In [6]:
def calcular_metricas_completas(retrieved_docs, relevant_docs):
    if not relevant_docs:
        return 0.0, 0.0, 0.0
    
    # 1. Average Precision (AP)
    relevant_count = 0
    precision_sum = 0.0
    for i, doc in enumerate(retrieved_docs, 1):
        if doc in relevant_docs:
            relevant_count += 1
            precision_at_i = relevant_count / i
            precision_sum += precision_at_i
    
    ap = precision_sum / len(relevant_docs) if relevant_count > 0 else 0.0
    
    # 2. Precisión y Sensibilidad (Recall) sobre todo el conjunto recuperado
    set_recuperados = set(retrieved_docs)
    set_relevantes = set(relevant_docs)
    aciertos = set_recuperados.intersection(set_relevantes)
    
    precision = len(aciertos) / len(set_recuperados) if set_recuperados else 0.0
    sensibilidad = len(aciertos) / len(set_relevantes) if set_relevantes else 0.0
    
    return ap, precision, sensibilidad

# Cargamos las necesidades de información
import pandas as pd
import numpy as np
df_necesidades = pd.read_csv('../necesidades_info/necesidades.csv')

lista_ap = []
lista_precision = []
lista_sensibilidad = []

print("\nEvaluando consultas vectoriales...\n")
print(f" {'ID':<3} | {'AP (%)':<8} | {'Prec (%)':<8} | {'Sens (%)':<8} | {'Consulta'}")
print("-" * 90)

for index, row in df_necesidades.iterrows():
    consulta_libre = row['consulta_texto_libre']
    relevantes = [doc.strip() for doc in row['relevantes'].split(',')]
    
    # Recuperamos documentos con similitud > 0
    resultados_busqueda = buscar_tfidf(consulta_libre)
    documentos_recuperados = [r['documento'] for r in resultados_busqueda]
    
    ap, precision, sensibilidad = calcular_metricas_completas(documentos_recuperados, relevantes)
    
    lista_ap.append(ap)
    lista_precision.append(precision)
    lista_sensibilidad.append(sensibilidad)
    
    print(f" Q{row['id']:<2} | {ap*100:>8.2f} | {precision*100:>8.2f} | {sensibilidad*100:>8.2f} | {consulta_libre}")

# Calculamos los promedios globales
MAP = np.mean(lista_ap)
mean_precision = np.mean(lista_precision)
mean_sensibilidad = np.mean(lista_sensibilidad)

print("\n" + "="*50)
print(f" MEAN AVERAGE PRECISION (MAP): {MAP*100:6.2f}%")
print(f" PRECISIÓN GLOBAL PROMEDIO:    {mean_precision*100:6.2f}%")
print(f" SENSIBILIDAD GLOBAL PROMEDIO: {mean_sensibilidad*100:6.2f}%")
print("="*50)



Evaluando consultas vectoriales...

 ID  | AP (%)   | Prec (%) | Sens (%) | Consulta
------------------------------------------------------------------------------------------
 Q1  |    23.33 |     3.03 |   100.00 | Swords Dance priority STAB sweeper cleaner
 Q2  |    26.59 |     3.00 |   100.00 | defensive pivot Defog Rapid Spin hazard removal
 Q3  |     7.07 |     2.00 |   100.00 | fast special attacker Fairy resistance check
 Q4  |    63.83 |     5.05 |   100.00 | Ground type Stealth Rock physical wall defensive
 Q5  |    38.59 |     3.00 |   100.00 | Knock Off special wall defensive utility
 Q6  |    57.69 |     3.03 |   100.00 | bulky Water type reliable recovery physical wall
 Q7  |   100.00 |     1.01 |   100.00 | Spore Regenerator pivot status
 Q8  |     4.55 |     2.00 |   100.00 | fast physical Dark Fighting Taunt attacker
 Q9  |   100.00 |     1.00 |   100.00 | Magic Bounce special attacker hazard control
 Q10 |    41.67 |     2.00 |   100.00 | fast Fire type Sun U-turn piv